# 03 · ONNX 流式导出 + Colab 侧评测 —— v2：DNS Challenge 真实噪声

跟 [v1 的 03](../v1_thchs30_musan/03_export_eval.ipynb) 逻辑完全一样——导出/校验/评测
代码本身不关心训练数据来源，只关心 `DRIVE_ROOT` 指向哪套 checkpoint/testset。
这里改的只有配置 cell（`CODE_ROOT`/`DRIVE_ROOT` 拆分），其余照抄。

In [ ]:
# ── 挂载 Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  配置 —— 所有路径与选择都集中在这一个 cell，别处不要再写死路径
#  这是 v2（DNS Challenge 真实噪声）。v1（THCHS-30+MUSAN）在
#  notebooks/v1_thchs30_musan/，两版数据/checkpoint 各自独立，不会互相覆盖。
# ═══════════════════════════════════════════════════════════════════════

# rtse-colab.zip 所在目录——**v1/v2 共用同一份代码包**，不需要重复上传。
# 如果你还没跑过 v1，这里跟下面 DRIVE_ROOT 的上一级目录填一样的就行。
CODE_ROOT = '/content/drive/MyDrive/Audio AI/RTSE'

# 本版本专属的数据/checkpoint/测试集根目录，嵌套在 CODE_ROOT 下面，
# 与 v1 直接写在 CODE_ROOT 下的 manifest.json/checkpoints/models/testset 互不冲突。
DRIVE_ROOT = '/content/drive/MyDrive/Audio AI/RTSE/v2_dns_real_noise'

# Colab 本地盘。**临时**，会话结束即消失，但读写比 Drive 快得多。
WORK_ROOT = '/content/rtse_work'

# ── 原始语料怎么放？────────────────────────────────────────────────────
#
#   'hybrid' —— **推荐**。压缩包缓存在 Drive，每次会话解压到本地盘。
#               一次下载永久有效；训练读取是本地盘全速；
#               每个新会话只需几分钟解压。
#
#   'local'  —— 全部放本地盘，压缩包用完即删。
#               不占 Drive；代价是**每次新会话都要重下**。
#
#   'drive'  —— 全部放 Drive。THCHS-30 一万多个小文件在 FUSE 挂载上解压很慢，
#               DNS 噪声/IR 是几个大文件（不是海量小文件）不受这个问题影响，
#               但仍不推荐——训练时随机读取 Drive 比本地盘慢。
DATA_MODE = 'hybrid'

# ── 快速验证模式 ────────────────────────────────────────────────────────
# True  = 只下载 337 MB 的小语料（LibriSpeech dev-clean，英文），不碰 DNS 数据，
#         约 15 分钟验证「数据→训练→导出→回传」整条链路。
#         **只用来验证流程，不要用它的结果做最终指标**。
# False = 完整流程：THCHS-30 中文语音 + DNS Challenge 真实噪声/RIR
QUICK_TEST = False

# ═══════════════════════════════════════════════════════════════════════

import os, sys, json, shutil, subprocess, time
from pathlib import Path
from shlex import quote as shq          # 路径里有空格时，所有 shell 命令都要靠它

assert DATA_MODE in ('hybrid', 'local', 'drive'), f'DATA_MODE 只能是 hybrid/local/drive'

CODE = CODE_ROOT
DRIVE = DRIVE_ROOT
WORK = WORK_ROOT

# 压缩包放哪 / 解压到哪 —— 三种模式的唯一区别就在这两行
ARCHIVE_DIR = f'{WORK}/archives' if DATA_MODE == 'local' else f'{DRIVE}/archives'
DATA = f'{DRIVE}/rawdata' if DATA_MODE == 'drive' else f'{WORK}/data'
KEEP_ARCHIVE = DATA_MODE != 'local'    # local 模式解压后删包省空间，其余保留以便复用

# Drive 侧的产物目录（**这些永远在 Drive 上**，训练结果不能放临时盘）
CKPT_DIR = f'{DRIVE}/checkpoints'   # 训练断点，每个 epoch 保存
MODEL_DIR = f'{DRIVE}/models'       # 导出的 ONNX
TESTSET_DIR = f'{DRIVE}/testset'    # 固定测试集
LOG_DIR = f'{DRIVE}/logs'

assert os.path.isdir(CODE), (
    f'Drive 上找不到 {CODE}\n'
    '检查两件事：① Drive 已挂载成功；② CODE_ROOT 与你实际存放 rtse-colab.zip 的目录一致。'
)
for d in [WORK, ARCHIVE_DIR, DATA, CKPT_DIR, MODEL_DIR, TESTSET_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('目录布局（v2 · DNS Challenge 真实噪声）')
print('─' * 74)
print(f'  代码包(v1/v2 共用)  {CODE}/rtse-colab.zip')
print(f'  压缩包缓存          {ARCHIVE_DIR}')
print(f'  语料解压目标        {DATA}')
print(f'  数据清单            {DRIVE}/manifest.json')
print(f'  固定测试集          {TESTSET_DIR}')
print(f'  训练断点  ★         {CKPT_DIR}/<模型名>/{{last,best}}.pt')
print(f'  导出模型  ★         {MODEL_DIR}/<模型名>.onnx')
print('─' * 74)
print(f'  数据模式  {DATA_MODE}')
print(f'  语料      {"快速验证(小语料/英文)" if QUICK_TEST else "完整流程(THCHS-30 中文语音 + DNS 真实噪声/RIR)"}')
print()

!df -h /content | tail -1
!df -h /content/drive 2>/dev/null | tail -1

In [ ]:
# ── 安装项目代码 ────────────────────────────────────────────────────────
# rtse-colab.zip 由本地 `uv run python scripts/pack_for_colab.py` 生成，
# 需要手动上传到 CODE_ROOT 指向的目录（v1/v2 共用同一份，不用重新上传）。
ZIP = f'{CODE}/rtse-colab.zip'
assert os.path.exists(ZIP), (
    f'找不到 {ZIP}\n'
    '请先在本地执行 `uv run python scripts/pack_for_colab.py`，'
    f'再把 dist/rtse-colab.zip 上传到 Drive 的 {CODE} 下。\n'
    f'该目录下现有：{sorted(os.listdir(CODE))[:12]}'
)

SRC = f'{WORK}/rtse-src'
shutil.rmtree(SRC, ignore_errors=True)
os.makedirs(SRC, exist_ok=True)
!unzip -q -o {shq(ZIP)} -d {shq(SRC)}

# 只装项目需要而 Colab 没有预装的几个包。
# 不用 `pip install -e .`：那会去解析 pyproject 里锁定的 torch CPU 索引，
# 把 Colab 自带的 GPU 版 torch 覆盖掉 —— 训练会瞬间慢几十倍。
!pip install -q soxr pystoi jiwer webrtcvad-wheels pesq onnx onnxruntime 2>&1 | tail -2

sys.path.insert(0, f'{SRC}/src')
import rtse
print('rtse', rtse.__version__, '| SR', rtse.SAMPLE_RATE, '| n_fft', rtse.N_FFT, '| hop', rtse.HOP_LENGTH)

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. 导出并校验

In [ ]:
import torch
from rtse.models import build_model
from rtse.train.export import export_streaming_onnx

MODELS = ['crn-nano', 'crn-lite']
export_info = {}

for name in MODELS:
    ck = Path(f'{CKPT_DIR}/{name}/best.pt')
    if not ck.exists():
        print(f'[skip] {name}: 没有 best.pt，先跑 02_train.ipynb'); continue

    state = torch.load(ck, map_location='cpu', weights_only=False)
    model = build_model(name)
    model.load_state_dict(state['model'])
    model.eval()

    info = export_streaming_onnx(model, f'{MODEL_DIR}/{name}.onnx', verify=True)
    v = info['verification']
    export_info[name] = info

    print(f'\n=== {name} ===')
    print(f"  训练到 epoch {state['epoch']}，最佳 val SI-SDR {-state['best_val']:.3f} dB")
    print(f"  参数 {info['params']:,}   文件 {info['size_kb']} KB")
    print(f"  ONNX流式 vs PyTorch整段 : {v['onnx_vs_pytorch_batch']:.3e} (相对 {v['relative_error']:.3e})")
    print(f"  PyTorch流式 vs 整段     : {v['pytorch_streaming_vs_batch']:.3e}")
    print(f"  状态形状稳定            : {v['state_shape_stable']}")
    print(f"  ==> {'PASS ✅' if v['passed'] else 'FAIL ❌ 不要下载这个模型，先查因果性'}")

Path(f'{MODEL_DIR}/export_info.json').write_text(
    json.dumps(export_info, ensure_ascii=False, indent=1), encoding='utf-8')

## 2. 在测试集上评测（含 PESQ）

`METHODS` 遍历同一份 testset 的全部样本，index.json 里每条记录带了 `noise_source`
字段（`real_dns` / `synthetic`），本地汇总时可以按这个字段分组，
直接对比"合成噪声上的指标"和"真实噪声上的指标"。

In [ ]:
import numpy as np
from tqdm.auto import tqdm
from rtse.audio.io import read_audio
from rtse.metrics.intrusive import si_sdr, stoi, estoi, pesq, seg_snr
from rtse.runtime import Pipeline, OnnxEnhancer
from rtse.dsp import build_dsp
from rtse.vad import build_vad

idx = json.loads(Path(f'{TESTSET_DIR}/index.json').read_text(encoding='utf-8'))
records = idx['records']
print(f'测试集 {len(records)} 个样本')

METHODS = ['none', 'specsub', 'wiener', 'mmse-lsa'] + list(export_info)

def make(method):
    if method == 'none': return None
    if method in ('specsub', 'wiener', 'mmse-lsa'): return build_dsp(method)
    return OnnxEnhancer(f'{MODEL_DIR}/{method}.onnx')

rows = []
for method in METHODS:
    for r in tqdm(records, desc=f'{method:>10}', leave=False):
        clean = read_audio(f'{TESTSET_DIR}/{r["clean"]}')
        noisy = read_audio(f'{TESTSET_DIR}/{r["noisy"]}')
        pipe = Pipeline(enhancer=make(method), vad=build_vad('energy'))
        enh, _ = pipe.process_signal(noisy)
        rows.append({'id': r['id'], 'method': method, 'snr': r['snr'],
                     'noise': r['noise'], 't60': r['t60'],
                     'noise_source': r.get('noise_source', 'synthetic'),
                     'si_sdr': si_sdr(clean, enh), 'seg_snr': seg_snr(clean, enh),
                     'stoi': stoi(clean, enh), 'estoi': estoi(clean, enh),
                     'pesq': pesq(clean, enh)})

Path(f'{DRIVE}/colab_metrics.json').write_text(json.dumps(rows, ensure_ascii=False), encoding='utf-8')
print(f'已写入 {len(rows)} 条指标 → {DRIVE}/colab_metrics.json')

In [ ]:
# 汇总看一眼，按 noise_source 分组——这是 v2 相对 v1 唯一多出来的对比维度
import collections, statistics as st
agg = collections.defaultdict(list)
for r in rows: agg[(r['method'], r['noise_source'])].append(r)

print(f"{'method':<12}{'noise_source':<12}{'SI-SDR':>9}{'STOI':>8}{'ESTOI':>8}{'PESQ':>8}")
print('-' * 57)
for m in METHODS:
    for src in ['synthetic', 'real_dns']:
        g = agg[(m, src)]
        if not g: continue
        pq = [r['pesq'] for r in g if r['pesq'] is not None]
        print(f"{m:<12}{src:<12}{st.mean(r['si_sdr'] for r in g):>9.2f}"
              f"{st.mean(r['stoi'] for r in g):>8.3f}"
              f"{st.mean(r['estoi'] for r in g):>8.3f}"
              f"{(st.mean(pq) if pq else float('nan')):>8.3f}")

## 3. 下载 DNSMOS 模型

跟 v1 一样，模型来自微软 DNS-Challenge 仓库，只有几 MB。

In [ ]:
os.makedirs(f'{MODEL_DIR}/dnsmos', exist_ok=True)
!wget -q -O "{MODEL_DIR}/dnsmos/sig_bak_ovr.onnx" \
  https://raw.githubusercontent.com/microsoft/DNS-Challenge/master/DNSMOS/DNSMOS/sig_bak_ovr.onnx \
  && ls -lh "{MODEL_DIR}/dnsmos/"

# 若 404，去 https://github.com/microsoft/DNS-Challenge 的 DNSMOS 目录确认最新路径。

## 4. 打包回传

跟 v1 一样，但下载到本地后放到**不同目录**，不要覆盖 v1 的产物：

| Drive 上的文件 | 放到本地 |
|---|---|
| `models/*.onnx` | `models/v2_dns/`（新建目录，跟 v1 的 `models/` 分开） |
| `colab_metrics.json` | `results/v2_dns_colab_metrics.json` |
| `testset.zip` | 解压到 `data/testset_v2/` |

In [ ]:
!cd "{DRIVE}" && rm -f colab_outputs.zip && zip -q -r colab_outputs.zip models colab_metrics.json && ls -lh colab_outputs.zip
print(f'从 Google Drive 下载 {DRIVE}/colab_outputs.zip 即可。')